# ch10 集成与融合（练习版）

> 按提示补全 `____`，跑通所有 assert。数据、划分与三个成员
> `members`（lr / svc / dtree，已拟合）已在脚手架就绪。
>
> 注意：挖空块内每条语句保持**单行**；循环/分支写在挖空块之外。

In [ ]:
from pathlib import Path

import warnings

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

import sklearn
from sklearn.ensemble import (AdaBoostClassifier, BaggingClassifier,
                              GradientBoostingClassifier,
                              RandomForestClassifier, StackingClassifier,
                              VotingClassifier)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

DATA = Path("data")

In [ ]:
# 数据与划分口径沿用 ch05：分层 8:2、median 填充、独热、标准化
dev = pd.read_csv(DATA / "defect_ml.csv")
NUM = ["负荷值", "温度", "湿度", "投运年限", "巡检耗时分钟"]
CAT = ["设备类型", "缺陷类型"]
TARGET = "是否危急"
y_all = dev[TARGET]

tr_idx, te_idx = train_test_split(np.arange(len(dev)), test_size=0.2,
                                  random_state=42, stratify=y_all)
dev_tr, dev_te = dev.iloc[tr_idx], dev.iloc[te_idx]
ys_tr, ys_te = dev_tr[TARGET], dev_te[TARGET]

imp = SimpleImputer(strategy="median").fit(dev_tr[NUM])
oh = OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit(dev_tr[CAT])
feat = NUM + [f"{c}_{v}" for c, vals in zip(CAT, oh.categories_) for v in vals]
Xtr = np.hstack([imp.transform(dev_tr[NUM]), oh.transform(dev_tr[CAT])])
Xte = np.hstack([imp.transform(dev_te[NUM]), oh.transform(dev_te[CAT])])
ss = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = ss.transform(Xtr), ss.transform(Xte)

# 三个异构成员先练好，后面 Voting/Stacking 直接复用
members = [("lr", LogisticRegression(max_iter=1000, random_state=42)),
           ("svc", SVC(probability=True, random_state=42)),
           ("dtree", DecisionTreeClassifier(max_depth=5, random_state=42))]
for _, est in members:
    est.fit(Xtr_s, ys_tr)

print("sklearn", sklearn.__version__, "| train", Xtr_s.shape,
      "| test", Xte_s.shape, "| 成员:", [name for name, _ in members])

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
def acc_of(model, Xt, yt):
    """一行算测试集 accuracy。"""
    return round(float(accuracy_score(yt, model.predict(Xt))), 4)


def staged_acc(model, Xt, yt):
    """staged_predict 的逐轮测试 accuracy 曲线。"""
    return [round(float(accuracy_score(yt, p)), 4)
            for p in model.staged_predict(Xt)]


print("脚手架就绪：acc_of / staged_acc")

## 任务 1：单树 vs Bagging

一棵全深树装进 50 棵树的袋子，看 acc 与 oob 的变化。

In [ ]:
# TODO(1)：单树 vs Bagging：一棵全深树装进 50 棵树的袋子里
# 提示：BaggingClassifier(DecisionTreeClassifier(random_state=42), n_estimators=50, oob_score=True, random_state=42)
tree = ____
tree_acc = ____
bag = ____
bag_acc = ____
bag_oob = ____
print("单树 acc:", tree_acc, "| 叶子数:", tree.get_n_leaves())
print("Bagging50 acc:", bag_acc, "| oob:", bag_oob)

assert tree.get_n_leaves() == 95, "全深树 95 叶，把训练集背得滚瓜烂熟"
assert tree_acc == 0.65, "单树泛化差：全深决策树是典型的高方差弱学习器"
assert bag_acc == 0.7833, "50 棵树投票后 +13.3 个点"
assert bag_oob == 0.7958, "oob 是免费的验证集：不用切数据就能估泛化"

## 任务 2：Bagging 数量曲线

10 / 25 / 100 棵树，验证「加树买稳定不买精度」。

In [ ]:
# TODO(2)：Bagging 数量曲线的骨架：建一个空字典存 {树数: acc}
# 提示：就一个空字典；下面的扫描循环已写好，不用挖
bag_scores = ____
for n in [10, 25, 100]:
    b = BaggingClassifier(DecisionTreeClassifier(random_state=42), n_estimators=n,
                          random_state=42, n_jobs=-1).fit(Xtr_s, ys_tr)
    bag_scores[n] = acc_of(b, Xte_s, ys_te)
print("数量→acc:", bag_scores)

assert bag_scores == {10: 0.7667, 25: 0.7833, 100: 0.775}
assert max(bag_scores.values()) - min(bag_scores.values()) <= 0.02, \
    "加树主要买的是稳定，不是精度：25 棵之后曲线躺平甚至回落"

## 任务 3：随机森林与特征重要性

RF100 训练，确认 max_features、acc / auc / oob 与 top3 特征。

In [ ]:
# TODO(3)：随机森林：Bagging + 每次分裂只看 sqrt(d) 个特征
# 提示：RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42)；重要性在 feature_importances_
rf = ____
rf_acc = ____
rf_auc = ____
top3 = ____
print("RF100 acc:", rf_acc, "| auc:", rf_auc, "| oob:", round(float(rf.oob_score_), 4))
print("内部单棵 max_features:", rf.estimators_[0].max_features, "| top3:", top3)

assert rf.estimators_[0].max_features == "sqrt", "16 特征的 sqrt≈4：RF 靠特征随机化去相关"
assert rf_acc == 0.7667 and rf_auc == 0.6872
assert round(float(rf.oob_score_), 4) == 0.8083, "oob 0.8083 比 test 0.7667 乐观——样本量不同，别拿 oob 交卷"
assert top3 == ["温度", "负荷值", "湿度"], "重要性与物理直觉一致：温度/负荷/湿度"

## 任务 4：AdaBoost staged 曲线

找出峰值轮数与最终轮分数的差——boosting 练过头的证据。

In [ ]:
# TODO(4)：AdaBoost：staged 曲线看『练过头』长什么样
# 提示：AdaBoostClassifier(n_estimators=100, random_state=42)；staged_acc(model, Xte_s, ys_te)
ada = ____
ada_curve = ____
ada_best_n = ____
ada_final = ____
w_head = ____
print("Ada 曲线峰值:", max(ada_curve), "@ 第", ada_best_n, "轮 | 最终:", ada_final)
print("前 5 轮基学习器权重:", w_head)

assert len(ada_curve) == 100
assert ada_best_n == 20 and max(ada_curve) == 0.8, "第 20 轮就到 0.80"
assert ada_final == 0.7667, "练到 100 轮反而回落——boosting 对噪声/过拟合不免疫"
assert w_head[0] == 1.3477 and w_head[2] == 0.2237, "第一棵树最重 1.3477；有的树几乎白学 0.2237"

## 任务 5：GradientBoosting 与 learning_rate

staged 峰值 + 三个学习率的敏感性实测。

In [ ]:
# TODO(5)：GradientBoosting：staged 峰值 + learning_rate 敏感性
# 提示：GradientBoostingClassifier(n_estimators=100, random_state=42) 先 fit；staged_acc(model, Xte_s, ys_te)
gb = ____
gb_curve = ____
gb_best_n = ____
lr_scores = {}
for lr in [0.05, 0.2, 0.5]:
    g = GradientBoostingClassifier(n_estimators=100, learning_rate=lr,
                                   random_state=42).fit(Xtr_s, ys_tr)
    lr_scores[lr] = acc_of(g, Xte_s, ys_te)
print("GB staged 峰值:", max(gb_curve), "@ 第", gb_best_n, "轮")
print("learning_rate → acc:", lr_scores)

assert len(gb_curve) == 100
assert gb_best_n == 15 and max(gb_curve) == 0.8, "GB 15 轮就封顶，比 Ada 的 20 轮还早"
assert lr_scores == {0.05: 0.775, 0.2: 0.775, 0.5: 0.7917}, \
    "本数据上 lr=0.5 反而最好——步子大不一定是坏事，lr 和 n_estimators 要一起调"

## 任务 6：手写 soft 投票

平均概率过阈值，与 `VotingClassifier(voting="soft")` 对账。

In [ ]:
# TODO(6)：手写 soft 投票 = VotingClassifier(voting="soft")：拆开看黑盒
# 提示：三个成员 predict_proba(Xte_s)[:, 1] 求平均，>=0.5 记 1
proba_avg = ____
hand_soft = ____
soft = ____
soft_acc = ____
agree = ____
print("手写平均概率 acc:", round(float(accuracy_score(ys_te, hand_soft)), 4),
      "| VotingClassifier soft acc:", soft_acc, "| 预测一致率:", agree)

assert soft_acc == 0.7917
assert agree == 1.0, "VotingClassifier soft 就是平均概率套阈值，没有任何魔法"

## 任务 7：加权投票

给最强成员 3 倍话语权，对比 hard 与等权 soft。

In [ ]:
# TODO(7)：加权投票：给最强成员 3 倍话语权
# 提示：VotingClassifier(estimators=members, voting="soft", weights=[1, 1, 3])
weighted = ____
w_acc = ____
hard = ____
hard_acc = ____
print("hard:", hard_acc, "| soft 等权:", 0.7917, "| soft w=[1,1,3]:", w_acc)

assert hard_acc == 0.7833 and w_acc == 0.8, "硬票 0.7833 → 等权软票 0.7917 → 加权 0.8"
assert w_acc > hard_acc, "给深 5 树的 dtree 加权后反超硬投票：成员质量不同就该区别对待"

## 任务 8：Stacking 元特征

确认元特征形状与元学习器系数，别掉进 6 维陷阱。

In [ ]:
# TODO(8)：Stacking：元特征形状与元学习器系数
# 提示：StackingClassifier(estimators=members, final_estimator=LogisticRegression(...), cv=5)；stack.transform(Xte_s) 看元特征
stack = ____
stack_acc = ____
stack_auc = ____
meta_shape = ____
coef = ____
print("元特征 shape:", meta_shape, "| final coef:", coef)
print("Stacking acc:", stack_acc, "| auc:", stack_auc)

assert meta_shape == (120, 3), "二分类 Stacking 元特征是各成员的正类概率，不是 6 维！"
assert coef == [2.8091, 1.4407, 0.4144], "lr 话语权最大 2.81，dtree 最小 0.41"
assert stack_acc == 0.8 and stack_auc == 0.7721, "本章冠军：0.80 / AUC 0.7721"

## 任务 9：成员多样性体检

两两一致率 + 手写多数票对账 hard voting。

In [ ]:
# TODO(9)：多样性体检：成员预测越不一致，融合越有肉吃
# 提示：两两一致率 np.mean(p1 == p2)；再手写多数票对账 hard voting
p_lr = ____
p_svc = ____
p_tree = ____
agree_ls = ____
agree_lt = ____
agree_st = ____
P = ____
hand_majority = ____
maj_acc = ____
print("一致率 lr~svc:", agree_ls, "| lr~tree:", agree_lt, "| svc~tree:", agree_st)
print("手写多数票 acc:", maj_acc, "（= hard voting 0.7833）")

assert agree_ls == 0.875 and agree_lt == 0.875 and agree_st == 0.8833, \
    "成员间仍有 11~12% 分歧——这正是融合能涨点的空间"
assert maj_acc == 0.7833, "手写多数票 == VotingClassifier(hard)，黑盒再次拆穿"

## 完成标准

所有 assert 通过 = 任务完成。自查：oob 和 test 哪个能写进报告、
boosting 什么时候该早停、Stacking 二分类元特征为什么是 3 列、
融合涨点的前提是什么。